## Part I. The Web Crawler
Let's start by building a web crawler that returns a sitemap, then we'll use `trafilatura` to deal with retrieving and deduplicating content.

We'll have three different versions: the first, simplest version will use a Python set and a queue (`from collections import deque`) for the visited set and "priority queue" respectively and we won't use `asyncio` yet. Version two will use a bloom filter and deal with `robots.txt`, I may also use a more sophisticated priority queue. Finally version three will be async with `asyncio`.

In [8]:
# !pip install pybloom-live

In [2]:
from bs4 import BeautifulSoup
from collections import deque
# from pybloom_live import ScalableBloomFilter
import requests
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser

In [3]:
class WebCrawler_v1:
    def __init__(self, startUrl: str):
        self.startUrl = startUrl

    def _is_crawlable(self, url: str) -> bool:
        parsed = urlparse(url)

        return parsed.scheme in ("http", "https") and bool(parsed.netloc)

    def _extract_links(self, url: str) -> list[str]:
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()

            soup = BeautifulSoup(response.text, "html.parser")

            links = []
            for tag in soup.find_all("a", href=True):
                href = tag["href"]
                absolute = urljoin(url, href)
                if self._is_crawlable(absolute):
                    links.append(absolute)

            return links
        except:
            return []

    def crawl(self, limit: int = 100) -> dict[str, list[str]]:
        queue = deque()
        visited = set([self.startUrl])
        sitemap = {}
        count = 0

        queue.append(self.startUrl)

        while queue and count <= limit:
            # pull the first item from the queue, add it to the sitemap,
            # extract links and append those to the queue
            # also iterate count
            url = queue.popleft()
            links = self._extract_links(url)
            sitemap[url] = links
            for link in links:
                if link not in visited:
                    queue.append(link)
                    visited.add(link)

            count += 1

        return sitemap

In [4]:
wc = WebCrawler_v1('https://pymotw.com/3/urllib.parse/index.html')
results = wc.crawl(limit = 10)

/tmp/ipykernel_21260/16322375.py:15: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(response.text, "html.parser")


In [5]:
print(results)

{'https://pymotw.com/3/urllib.parse/index.html': ['https://pymotw.com/3/index.html', 'https://pymotw.com/3/py-modindex.html', 'https://pymotw.com/3/genindex.html', 'http://www.twitter.com/pymotw', 'https://feeds.feedburner.com/PyMOTW', 'https://pymotw.com/3/internet_protocols.html', 'https://pymotw.com/3/urllib.parse/index.html#module-urllib.parse', 'https://pymotw.com/3/urllib.parse/index.html#parsing', 'https://pymotw.com/3/urllib.parse/index.html#id1', 'https://pymotw.com/3/urllib.parse/index.html#id2', 'https://tools.ietf.org/html/rfc2396.html', 'https://pymotw.com/3/urllib.parse/index.html#id3', 'https://pymotw.com/3/urllib.parse/index.html#id4', 'https://pymotw.com/3/urllib.parse/index.html#unparsing', 'https://pymotw.com/3/urllib.parse/index.html#id5', 'https://pymotw.com/3/urllib.parse/index.html#id6', 'https://pymotw.com/3/urllib.parse/index.html#id7', 'https://pymotw.com/3/urllib.parse/index.html#joining', 'https://pymotw.com/3/urllib.parse/index.html#id8', 'https://pymotw.co

In [6]:
len(results['https://pymotw.com/3/urllib.parse/index.html'])

53

In [7]:
len(results)

11

In [ ]:
class WebCrawler_v2:
    def __init__(self, startUrl: str):
        self.startUrl = startUrl
        self.robots_cache = {}
        self.allowed_domain = urlparse(startUrl).netloc

    def _is_same_domain(self, url: str) -> bool:
        return urlparse(url).netloc == self.allowed_domain

    def _is_crawlable(self, url: str) -> bool:
        parsed = urlparse(url)
        return parsed.scheme in ["http", "https"] and bool(parsed.netloc)

    def _get_robots(self, url: str) -> RobotFileParser:
        parsed = urlparse(url)
        base = f"{parsed.scheme}://{parsed.netloc}"

        if base not in self.robots_cache:
            rp = RobotFileParser()
            rp.set_url(f"{base}/robots.txt")
            try:
                rp.read()
            except Exception:
                pass # if robots.txt not found, assume allowed
            self.robots_cache[base] = rp

        return self.robots_cache[base]

    def _is_allowed(self, url: str) -> bool:
        rp = self._get_robots(url)
        return rp.can_fetch("*", url)

    def _extract_links(self, url: str) -> list[str]:
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()

            soup = BeautifulSoup(response.text, "html.parser")

            links = []

            for tag in soup.find_all("a", href=True):
                href = tag["href"]
                absolute = urljoin(url, href)
                if self._is_crawlable(absolute):
                    links.append(absolute)

            return links
        except:
            return []

    def crawl(self, limit: int = 1000) -> dict[str, list[str]]:
        queue = deque()
        # visited = set([self.startUrl])
        visited = ScalableBloomFilter(mode=ScalableBloomFilter.SMALL_SET_GROWTH, error_rate=0.001)
        sitemap = {}
        count = 0

        visited.add(self.startUrl)
        queue.append(self.startUrl)

        while queue and count < limit:
            url = queue.popleft()
            if not self._is_allowed(url):
                continue
            ## we need to do something with Crawl-delay probably

            links = self._extract_links(url)
            sitemap[url] = links
            for link in links:
                if link not in visited and self._is_same_domain(link):
                    queue.append(link)
                    visited.add(link)

            count += 1

        return sitemap

In [ ]:
wc2 = WebCrawler_v2('https://pymotw.com/3/urllib.parse/index.html')
results = wc2.crawl(limit=10)
print(results)

In [ ]:
len(results)

10

In [ ]:
class WebCrawler_v3:
    def __init__(self, startUrl: str):
        self.startUrl = startUrl

    def crawl(self, limit: int = 1000) -> dict[str, list[str]]:
        pass

1. Use trafilatura for proper extraction
2. Deduplication with MinHash LSH for fuzzy/near-duplicate detection
3. SHA-256 for exact dedup

In [1]:
#!pip install trafilatura

In [2]:
#from trafilatura import fetch_url, extract

In [ ]:
from urllib.robotparser import RobotFileParser
rp = RobotFileParser()
rp.set_url('https://pymotw.com/3/urllib.parse/index.html')
rp.read()

In [ ]:
rp.can_fetch('*', 'https://pymotw.com/3/urllib.parse/index.html#parsing')

True

In [ ]:
rp.can_fetch('*', 'https://pymotw.com/3/internet_protocols.html')

True

In [ ]:
rp.crawl_delay('*')